In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src.pannuke_dataset import PannukePreparedDataset
from models.lit_unet_pannuke import UNetLightning
import ioumatch
from skimage.measure import label as sklabel

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

BASE_DIR = Path("/run/user/1000/gvfs/smb-share:server=zeus.pasteur.fr,share=bia/ayehadji/projet0")
DATA_ROOT = BASE_DIR / "data" / "prepared" / "pannuke"
CKPT_DIR = PROJECT_ROOT / "checkpoints"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

ckpts = sorted(CKPT_DIR.glob("unet_pannuke_lit_best.ckpt"))
print("Checkpoint chargé :", ckpt_path)

model = UNetLightning.load_from_checkpoint(ckpt_path).to(device)
model.eval()

ds_test = PannukePreparedDataset(
    root=DATA_ROOT,
    split="test",
    transform=None,
    return_image_name=True,
)

df_features = pd.read_csv(DATA_ROOT / "pannuke_morphology_with_classes.csv")
df_features["split"].value_counts(), df_features["morph_class"].value_counts()


Device: cuda
Checkpoint chargé : /home/bia/PycharmProjects/project0/checkpoints/unet_pannuke_lit_best.ckpt


(split
 test     58123
 train    55569
 val      52754
 Name: count, dtype: int64,
 morph_class
 elongated    70637
 other        47662
 round        39211
 irregular     8936
 Name: count, dtype: int64)

In [2]:
# def bin_to_instances(pred_bin: np.ndarray):
#     return sklabel(pred_bin.astype(np.uint8))

def bin_to_instances(pred_prob: np.ndarray, thr=0.5):
    return prob_to_instances(pred_prob, thr=thr, min_size=10)

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.measure import label
from skimage.morphology import remove_small_objects

def prob_to_instances(prob, thr=0.5, min_size=10):
    # prob: (H,W) en [0,1]
    bin_mask = prob > thr

    # nettoyage de bruit
    bin_mask = remove_small_objects(bin_mask, min_size=min_size)

    # distance transform (plus grand au centre des blobs)
    dist = ndi.distance_transform_edt(bin_mask)

    # pics locaux dans le dist
    local_max = (dist == ndi.maximum_filter(dist, size=5))
    markers, _ = ndi.label(local_max)

    # watershed
    labels_ws = watershed(-dist, markers, mask=bin_mask)

    return labels_ws.astype(np.int32)


def extract_pairs_from_iou_matrix(M, labels_pred, labels_gt, threshold=0.5, inclusive=False):
    if inclusive:
        valid = M >= threshold
    else:
        valid = M > threshold

    pred_idx, gt_idx = np.where(valid)
    order = np.argsort(M[pred_idx, gt_idx])[::-1]

    used_pred = set()
    used_gt = set()
    pairs = []

    for k in order:
        pi = int(pred_idx[k])
        gi = int(gt_idx[k])
        iou = float(M[pi, gi])

        if pi in used_pred or gi in used_gt:
            continue

        used_pred.add(pi)
        used_gt.add(gi)

        pred_label = int(labels_pred[pi])
        gt_label = int(labels_gt[gi])

        pairs.append(
            {
                "pred_label": pred_label,
                "gt_label": gt_label,
                "iou": iou,
            }
        )

    return pairs


def match_pred_gt(pred_inst: np.ndarray, gt_inst: np.ndarray, thr: float = 0.5):
    labels_pred = np.unique(pred_inst)
    labels_pred = labels_pred[labels_pred > 0]

    labels_gt = np.unique(gt_inst)
    labels_gt = labels_gt[labels_gt > 0]

    res = ioumatch.evaluate_image(
        pred_inst,
        gt_inst,
        threshold=thr,
        method="greedy",
        inclusive=False,
        normalize=False,
    )
    M = res["iou_matrix"]

    if M.size == 0 or len(labels_gt) == 0:
        return [], set(), set()

    pairs = extract_pairs_from_iou_matrix(M, labels_pred, labels_gt, threshold=thr)

    gt_labels_all = set(labels_gt)
    gt_labels_matched = {p["gt_label"] for p in pairs}

    return pairs, gt_labels_all, gt_labels_matched


def get_morphology_map(df, split, image_name):
    dfi = df[(df["split"] == split) & (df["image"] == image_name)]
    if dfi.empty:
        return {}
    return dict(zip(dfi["label"], dfi["morph_class"]))


from collections import defaultdict


def evaluate_image_morphology(model, img, mask_inst, morph_map, device, thr=0.5):
    gt_inst = mask_inst.numpy().astype(np.int32)

    with torch.no_grad():
        logits = model(img.unsqueeze(0).to(device))
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    pred_bin = (prob > thr).astype(np.uint8)
    pred_inst = bin_to_instances(pred_bin)

    pairs, gt_labels_all, gt_labels_matched = match_pred_gt(pred_inst, gt_inst, thr=thr)

    per_class_iou = defaultdict(list)
    per_class_total = defaultdict(int)
    per_class_matched = defaultdict(int)

    for lbl in gt_labels_all:
        morph = morph_map.get(lbl, "unknown")
        per_class_total[morph] += 1

    for p in pairs:
        gt_label = p["gt_label"]
        iou = p["iou"]
        morph = morph_map.get(gt_label, "unknown")

        per_class_iou[morph].append(iou)
        per_class_matched[morph] += 1

    return per_class_iou, per_class_total, per_class_matched


def evaluate_split_morphology(model, df_features, device, dataset, split_name: str, thr=0.5):
    summary_iou = defaultdict(list)
    summary_total = defaultdict(int)
    summary_matched = defaultdict(int)

    for img, mask_inst, name in dataset:
        morph_map = get_morphology_map(df_features, split_name, name)

        per_iou, per_tot, per_match = evaluate_image_morphology(
            model, img, mask_inst, morph_map, device, thr=thr
        )

        for cls, values in per_iou.items():
            summary_iou[cls].extend(values)
        for cls, n in per_tot.items():
            summary_total[cls] += n
        for cls, n in per_match.items():
            summary_matched[cls] += n

    summary = {}
    for cls in summary_total.keys():
        ious = np.array(summary_iou.get(cls, []))
        total = summary_total[cls]
        matched = summary_matched.get(cls, 0)
        coverage = matched / total if total > 0 else 0.0
        mean_iou = ious.mean() if len(ious) > 0 else 0.0
        std_iou = ious.std() if len(ious) > 0 else 0.0

        summary[cls] = {
            "mean_iou": mean_iou,
            "std_iou": std_iou,
            "n_tp": matched,
            "n_gt": total,
            "coverage": coverage,
        }

    return summary


In [3]:
idx = 0
img, mask_inst, name = ds_test[idx]

labels_in_mask = np.unique(mask_inst.numpy())
labels_in_mask = labels_in_mask[labels_in_mask > 0]
print("Nb labels dans mask      :", len(labels_in_mask))

dfi = df_features[(df_features["split"] == "test") & (df_features["image"] == name)]
labels_in_csv = dfi["label"].unique()
print("Nb labels dans df_features:", len(labels_in_csv))

print("Exemple labels mask:", labels_in_mask[:20])
print("Exemple labels csv :", labels_in_csv[:20])


Nb labels dans mask      : 13
Nb labels dans df_features: 13
Exemple labels mask: [ 1  2  3  4  5  6  7  8  9 10 11 12 13]
Exemple labels csv : [ 1  2  3  4  5  6  7  8  9 10 11 12 13]


In [4]:
summary = evaluate_split_morphology(
    model=model,
    df_features=df_features,
    device=device,
    dataset=ds_test,
    split_name="test",
    thr=0.5,
)

for cls, stats in summary.items():
    print(
        f"{cls:10s}  "
        f"mean_iou={stats['mean_iou']:.3f}  "
        f"std={stats['std_iou']:.3f}  "
        f"TP={stats['n_tp']}  "
        f"GT={stats['n_gt']}  "
        f"coverage={stats['coverage']:.3f}"
    )


round       mean_iou=0.841  std=0.109  TP=10518  GT=13597  coverage=0.774
elongated   mean_iou=0.806  std=0.118  TP=17009  GT=24665  coverage=0.690
other       mean_iou=0.829  std=0.112  TP=12434  GT=16630  coverage=0.748
irregular   mean_iou=0.739  std=0.132  TP=1896  GT=3228  coverage=0.587


In [6]:
# apres post processing
print("apres post processing ")
summary = evaluate_split_morphology(
    model=model,
    df_features=df_features,
    device=device,
    dataset=ds_test,
    split_name="test",
    thr=0.5,
)

for cls, stats in summary.items():
    print(
        f"{cls:10s}  "
        f"mean_iou={stats['mean_iou']:.3f}  "
        f"std={stats['std_iou']:.3f}  "
        f"TP={stats['n_tp']}  "
        f"GT={stats['n_gt']}  "
        f"coverage={stats['coverage']:.3f}"
    )


apres post processing 
round       mean_iou=0.841  std=0.109  TP=10518  GT=13597  coverage=0.774
elongated   mean_iou=0.806  std=0.118  TP=17009  GT=24665  coverage=0.690
other       mean_iou=0.829  std=0.112  TP=12434  GT=16630  coverage=0.748
irregular   mean_iou=0.739  std=0.132  TP=1896  GT=3228  coverage=0.587
